In [ ]:
#Importing numpy and pandas libraries
#Numpy for math
import numpy as np
#Pandas will be used for the dataset
import pandas as pd

In [ ]:
#Path to the dataset
file_path = r"C:\Users\adbar\Downloads\MLDataset\spam_email_dataset.csv"

#Reading the csv using pandas
df = pd.read_csv(file_path)
#Proof that our dataset is loaded successfully
print("Dataset loaded successfully.")
#Printing the shap of the dataset and the first 5 rows
print("Shape:", df.shape)
df.head()

In [ ]:
#Here we will check the columns, data types, and missing values in the dataset
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
#Here we will drop the missing values and convert categorical variables into dummy variables (converting non numerical data into numerical data)
df = df.dropna()
df = pd.get_dummies(df, drop_first=True)

print("New shape after cleaning:", df.shape)
print("New columns:")
print(df.columns.tolist())

In [ ]:
#Here we will separate the features and the target variable (what we want it to predict, the price)
target_column = "price"

#Here we will convert the features and the target variable into numpy arrays and check their dimensions
X = df.drop(columns=[target_column]).values.astype(float)
y = df[target_column].values.astype(float)

#Here we will print the shapes of the features and the target variable to confirm that they are correct
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
#Funtion to add a bias column to the features because in linear regression we need to account for the intercept term (the bias) and adding a column of ones allows us to do that
def add_bias_column(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack((ones, X))

In [ ]:
#Function to standardize the features using the mean and standard deviation of the training set
#Putting all of the feautres (data) on the same scale, dont want one feature to dominate others because of its scale
def standardize_train_val(X_train, X_val):
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    
    # Prevent division by zero
    std[std == 0] = 1
    
    #Standardize the training and validation sets using the mean and standard deviation of the training set
    X_train_scaled = (X_train - mean) / std
    X_val_scaled = (X_val - mean) / std
    
    #Return the standardized training and validation sets
    return X_train_scaled, X_val_scaled

In [ ]:
#Our funtion to fit the ridge regression model using the closed-form solution
#Finding the weights that minimize the cost
def ridge_fit(X, y, lam):
    #Setting the number of features to the number of columns in X
    n_features = X.shape[1]
    
    #Creating the identity matrix and setting the first element to 0 to avoid regularizing the bias term
    I = np.eye(n_features)
    I[0, 0] = 0   # do not regularize bias
    
    #Calculating the weights using the closed-form solution of ridge regression
    w = np.linalg.inv(X.T @ X + lam * I) @ X.T @ y
    return w

In [11]:
#Function to predict the target variable using the features and the weights
def predict(X, w):
    return X @ w

In [12]:
#Function to calculate the mean squared error between the true and predicted values
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

In [ ]:
#Function to create k folds for cross-validation (We just need 5)
#5 partitions of the dataset
def create_folds(X, y, k=5, seed=42):
    #Setting the random seed for reproducibility
    np.random.seed(seed)
    
    #Shuffling the indices of the dataset
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    
    #Splitting the shuffled dataset into k folds
    X_shuffled = X[indices]
    y_shuffled = y[indices]
    
    X_folds = np.array_split(X_shuffled, k)
    y_folds = np.array_split(y_shuffled, k)
    
    #Returning the folds of the features and the target variable, features should stay the same across the folds
    return X_folds, y_folds

In [14]:
#Function to perform k-fold cross-validation for ridge regression and return the average mean squared error across the folds
def cross_validate_ridge(X, y, lam, k=5):
    #Creating the folds for cross-validation
    X_folds, y_folds = create_folds(X, y, k=k)
    errors = []
    
    #Iterating through each fold to use it as the validation set and the rest as the training set
    for i in range(k):
        #The current fold will be the validation set and the rest will be the training set
        X_val = X_folds[i]
        y_val = y_folds[i]

        #Combining the other folds to create the training set
        X_train = np.vstack([X_folds[j] for j in range(k) if j != i])
        y_train = np.hstack([y_folds[j] for j in range(k) if j != i])
    
        # Standardize correctly using only training data
        X_train_scaled, X_val_scaled = standardize_train_val(X_train, X_val)
        
        # Add bias column
        X_train_scaled = add_bias_column(X_train_scaled)
        X_val_scaled = add_bias_column(X_val_scaled)
        
        # Train
        w = ridge_fit(X_train_scaled, y_train, lam)
        
        # Predict
        y_pred = predict(X_val_scaled, w)
        
        # Evaluate
        error = mse(y_val, y_pred)
        errors.append(error)
    
    return np.mean(errors)

In [ ]:
#Function to perform grid search over a list of lambda values and return the best lambda and its corresponding average mean squared error
#Lambdas will be tested across folds 
def grid_search_ridge(X, y, lambda_values, k=5):
    #initializing best lambda variable
    best_lambda = None
    #Intiializing the best score variable
    best_score = float("inf")
    #Initializng the result list
    results = []
    
    #For every lam in the list of lambda values
    for lam in lambda_values:
        #Callin the cross-validation function to get the average mean squared error for the current lambda value
        avg_mse = cross_validate_ridge(X, y, lam, k=k)
        #Adding the lambda value and its average mean to the result list
        results.append((lam, avg_mse))
        
        #Printing out the lamnda and the average mean
        print(f"Lambda = {lam}, Average CV MSE = {avg_mse:.4f}")
        
        #If the average mean is less than the best score we have seen so far, well update the values
        if avg_mse < best_score:
            best_score = avg_mse
            best_lambda = lam
    
    #Returning the best lambda and its score
    return best_lambda, best_score, results

In [ ]:
#Setting a list of lambda values to search over
lambda_values = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000]

#Calculate the grid search function to find the best lambda and its average mean squared error
best_lambda, best_score, results = grid_search_ridge(X, y, lambda_values, k=5)

#Printing the best lambda and the best average mean squared error
print("\nBest lambda:", best_lambda)
print("Best average CV MSE:", best_score)

In [ ]:
#Creating a data frame to display all of the results
results_df = pd.DataFrame(results, columns=["Lambda", "Average_CV_MSE"])
results_df

In [ ]:
# Standardize the full dataset
mean = np.mean(X, axis=0)
std = np.std(X, axis=0)
std[std == 0] = 1

X_scaled = (X - mean) / std
X_scaled = add_bias_column(X_scaled)

# Train final ridge model
final_weights = ridge_fit(X_scaled, y, best_lambda)

#Printing the final results of the model trained on the entire dataset with the best lambda
print("Final model trained.")
print("Best lambda used:", best_lambda)
print("Number of weights:", len(final_weights))
print("First 10 weights:", final_weights[:10])

In [ ]:
#Setting the final mean squared error on the full dataset using the final model
y_pred_full = predict(X_scaled, final_weights)
final_mse = mse(y, y_pred_full)
#Printing the final mean squared error on the full dataset
print("Final training MSE on full dataset:", final_mse)

In [ ]:
#Here we are calculating the root mean squared error to have a more interpretable metric of the model's performance on the full dataset
#RMSE is MSE expressed in the origianl units
rmse = np.sqrt(final_mse)
print("Final training RMSE:", rmse)